In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, urllib.request

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/CameraTrap_Phase2')
source = root / 'phase2_results/final_sprint_v1/code'
assert (source/'runtime.py').is_file(), f'Mount the Drive containing {source}'
print('Preparing the runtime...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(source/'requirements.txt')], check=True)
local = Path(tempfile.mkdtemp(prefix='phase2_candidate_'))
revision = '9552a744291de5d7f8f7cf44a504788e7528a57e'
checksums = {'candidate.py': '6ded6d4de21f5d615d494f80e661500b48c897a0cbfe28859d44792873cefa44', 'crop_sam.py': '74d5a18a2081d1c62dcf3b7d7fd4e4d8a1c93b82d3673209eb5cb9051a04844a', 'run_candidate.py': '722c284420cfbd77d73e1ce4ae30e4a14b284e08b7fe254125a00c4ab40167c2'}
for name, expected in checksums.items():
    url = f'https://raw.githubusercontent.com/salehdev3/Capstone_5/{revision}/experiments/fast_candidate/{name}'
    with urllib.request.urlopen(url, timeout=60) as stream:
        payload = stream.read()
    assert hashlib.sha256(payload).hexdigest() == expected, name
    (local/name).write_bytes(payload)
env = dict(os.environ, PYTHONPATH=str(source), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
subprocess.run([sys.executable, '-u', str(local/'run_candidate.py'), str(root)], env=env, check=True)
